# 1. Save GraphCast activations to Google Drive

**Run this notebook first. No code changes are needed for your first run.**

1. Open this in **Google Colab in your browser**. Choose **Runtime → Change runtime type → A100 GPU** (or H100). Use a high-RAM runtime if available.
2. Choose **Runtime → Run all**, and approve the Google Drive connection when asked.
3. Wait for **“Finished”**, then open [notebook 2: Train TopK SAE](https://colab.research.google.com/github/observerhan/graphcast-interpretability/blob/simple-colab/02_train_topk_sae.ipynb).

The notebook downloads public weather data, runs the pretrained **full 0.25° GraphCast**, and saves **layer-8 mesh activations** directly in `My Drive/AMS691_GraphCast/activations_layer8`.
Each file has 40,962 rows and 512 columns and uses about 80 MiB. The default four files use about 320 MiB.
These files remain in Drive after Colab disconnects. GraphCast itself is **not trained**.

**Hardware:** full GraphCast needs a large GPU; a standard free T4 is not sufficient for this workflow. The first forecast also compiles the model, so it is much slower than later forecasts. Setup and weather downloads may take several minutes each.

## Step 1 — Connect your Google Drive

Use the same Google account in both notebooks. The folder is created for you.

In [ ]:
# @title Connect Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT_FOLDER = Path('/content/drive/MyDrive/AMS691_GraphCast')
ACTIVATION_DIR = PROJECT_FOLDER / 'activations_layer8'
ACTIVATION_DIR.mkdir(parents=True, exist_ok=True)
print('Your project folder:', PROJECT_FOLDER)

## Step 2 — Choose the dates (the defaults are ready to use)

We collect one snapshot at **12:00 UTC on each date**. Four dates are enough to check that the whole workflow works. This is a small starting dataset, not the paper's full training dataset.

Later, you can add dates here and rerun. Files already saved are checked and skipped. Use dates from **1959–2022**, the range of the public dataset.

In [ ]:
# @title Dates to extract
import datetime
import json

DATES = ['2021-08-27', '2021-08-28', '2021-08-29', '2021-08-30']
DATES = sorted(set(datetime.date.fromisoformat(day).isoformat() for day in DATES))
if len(DATES) < 2:
    raise ValueError('Keep at least two dates so notebook 2 can reserve one for validation.')
CONFIG_PATH = Path('/content/ams691_extraction_settings.json')
CONFIG_PATH.write_text(json.dumps({'dates': DATES, 'activation_dir': str(ACTIVATION_DIR)}))
print('Dates:', ', '.join(DATES))
print('Activations will be saved to:', ACTIVATION_DIR)

## Step 3 — Set up GraphCast automatically

Run this cell unchanged. You do not need to install anything on your laptop or select package versions.

In [ ]:
# @title Automatic GraphCast setup (run unchanged; takes several minutes)
import os
import shutil
import subprocess
import sys
from pathlib import Path

if not shutil.which('nvidia-smi'):
    raise RuntimeError('Choose Runtime > Change runtime type > A100 GPU, then run again.')
gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
    capture_output=True, text=True,
)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('Choose Runtime > Change runtime type > A100 GPU, then run again.')
print('GPU:', gpu.stdout.strip())
memory_mb = float(gpu.stdout.strip().splitlines()[0].rsplit(',', 1)[1])
if memory_mb < 30000:
    raise RuntimeError('This notebook uses full GraphCast. Select an A100 or H100 GPU; the usual T4 has too little memory.')

# GraphCast runs in its own automatically prepared Python environment.
# This avoids changing Colab's imported packages or requiring a kernel restart.
ENV_DIR = Path('/content/ams691_graphcast_env')
PYTHON = str(ENV_DIR / 'bin/python')

def setup_command(command):
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode:
        print(result.stdout[-4000:])
        print(result.stderr[-8000:])
        raise RuntimeError('Setup stopped. The installation error is printed above.')

if not Path(PYTHON).exists():
    setup_command([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv==20.31.2'])
    setup_command([sys.executable, '-m', 'virtualenv', '--python', sys.executable, str(ENV_DIR)])

print('Installing the packages GraphCast needs...')
packages = [
    'jax[cuda12]==0.4.38', 'dm-haiku==0.0.13', 'chex==0.1.88',
    'numpy==2.2.6', 'pandas==2.2.3', 'scipy==1.15.3', 'xarray==2025.1.2',
    'zarr==2.18.7', 'numcodecs==0.15.1', 'gcsfs==2025.3.0',
    'google-cloud-storage==3.1.0', 'dask[array]==2025.3.0',
    'dm-tree==0.1.9', 'jraph==0.0.6.dev0', 'trimesh==4.6.8',
    'rtree==1.4.0', 'netCDF4==1.7.2',
]
setup_command([PYTHON, '-m', 'pip', 'install', '-q', *packages])
setup_command([PYTHON, '-m', 'pip', 'install', '-q', '--no-deps',
               'git+https://github.com/theodoremacmillan/graphcast.git@39d1de436148c5658726b5a092bd2d7ef8701f2a'])

# The authors' callback copies every layer to the host before filtering.
# Filter first so only the selected layer-8 mesh array is transferred.
patch_code = r"""
import importlib.util
from pathlib import Path
root = Path(next(iter(importlib.util.find_spec('graphcast').submodule_search_locations)))
path = root / 'deep_typed_graph_net.py'
old = '            tag = f"{self.name}/post_res/nodes/{nset}"\n            jdebug.callback('
new = '            tag = f"{self.name}/post_res/nodes/{nset}"\n            if not am._should_save(tag, step_idx, nset):\n                continue\n            jdebug.callback('
source = path.read_text()
if old in source:
    assert source.count(old) == 1
    path.write_text(source.replace(old, new))
elif new not in source:
    raise RuntimeError('Unexpected GraphCast hook source; cannot apply the memory-saving change.')
from graphcast import graphcast, data_utils
import jax
assert jax.__version__ == '0.4.38'
assert any(device.platform == 'gpu' for device in jax.devices()), 'JAX cannot see a GPU'
print('GraphCast imports and GPU check passed.')
"""
setup_command([PYTHON, '-c', patch_code])
print('Setup complete. Continue to the next cell; no restart is needed.')

## Step 4 — Prepare the extraction code

Run this cell unchanged. It prepares the same weather-model workflow as the original hurricane demo, with automatic downloading and Google Drive saving.
The weather data immediately before and after each selected time are included automatically. Only the activations need to be kept permanently.

In [ ]:
%%writefile /content/ams691_extract.py
# @title Prepare the extraction program (run unchanged)
"""This program is embedded in notebook 1; no separate file is needed by the user."""
import dataclasses
import gc
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import haiku as hk
import jax
import numpy as np
import xarray as xr
from google.cloud import storage
from graphcast import autoregressive, casting, checkpoint, data_utils
from graphcast import graphcast, normalization
from graphcast.deep_typed_graph_net import get_activation_manager

GRAPHCAST_COMMIT = "39d1de436148c5658726b5a092bd2d7ef8701f2a"
MODEL_NAME = (
    "GraphCast - ERA5 1979-2017 - resolution 0.25 - pressure levels 37 "
    "- mesh 2to6 - precipitation input and output.npz"
)
WEATHER_STORE = "gs://weatherbench2/datasets/era5/1959-2022-full_37-6h-0p25deg_derived.zarr"


def write_json(path, value):
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(value, indent=2) + "\n")
    temporary.replace(path)


def check_array(path, expected_shape=(40962, 512)):
    array = np.load(path, mmap_mode="r", allow_pickle=False)
    if array.shape != expected_shape or array.dtype != np.float32:
        raise ValueError(f"{path.name}: expected float32 {expected_shape}, got {array.dtype} {array.shape}")
    if not np.isfinite(array).all():
        raise ValueError(f"{path.name} contains invalid numbers. Remove this file and rerun extraction.")
    return array.shape


def forecast_window(source, center, pressure_levels):
    # Read the two input times and the next six-hour target/forcing time.
    # This also works when the window crosses midnight.
    times = np.datetime64(center, "ns") + np.array([-6, 0, 6], dtype="timedelta64[h]")
    window = source.sel(time=times, level=list(pressure_levels)).load()
    for name in window.data_vars:
        if "time" in window[name].dims:
            window[name] = window[name].expand_dims(batch=[0])
    return window.assign_coords(
        time=times - times[0], datetime=(("batch", "time"), times[None, :])
    )


def install_writer(manager, directory, expected_shape=(40962, 512)):
    manager.__init__(enabled=True, save_dir=str(directory), save_steps=[8],
                     save_node_sets=["mesh_nodes"], mode="post_res")

    def save_float32(tag, x, *, step_idx=None, node_set=None, time_str=None):
        if not manager._should_save(tag, step_idx, node_set):
            return
        timestamp = time_str or manager.current_time_str
        if timestamp is None:
            raise ValueError("The forecast time has not been set.")
        # Numerically convert bfloat16; do not reinterpret its bytes as float16.
        array = np.asarray(x).astype(np.float32)
        if array.ndim == 3 and array.shape[1] == 1:
            array = array[:, 0, :]
        if array.shape != expected_shape or not np.isfinite(array).all():
            raise ValueError(f"Unexpected activation values or shape: {array.shape}")
        path = directory / f"layer0008_t{timestamp}.npy"
        if path.exists():
            raise FileExistsError(f"Refusing to overwrite an existing snapshot: {path}")
        temporary = path.with_suffix(".npy.tmp")
        with temporary.open("wb") as handle:
            np.save(handle, array, allow_pickle=False)
        temporary.replace(path)
        print(f"Saved to Drive: {path.name}  {array.shape}", flush=True)

    manager.save = save_float32


def make_forward(model_config, task_config, stats):
    @hk.transform_with_state
    def forward(inputs, targets_template, forcings):
        predictor = graphcast.GraphCast(model_config, task_config)
        predictor = casting.Bfloat16Cast(predictor)
        predictor = normalization.InputsAndResiduals(predictor, **stats)
        predictor = autoregressive.Predictor(predictor, gradient_checkpointing=True)
        return predictor(inputs, targets_template=targets_template, forcings=forcings)
    return forward


def main(config_path):
    config = json.loads(Path(config_path).read_text())
    directory = Path(config["activation_dir"])
    directory.mkdir(parents=True, exist_ok=True)
    centers = [f"{day}T12" for day in config["dates"]]
    if len(set(config["dates"])) < 2 or len(set(centers)) != len(centers):
        raise ValueError("Choose at least two different dates, with no duplicates.")
    for center in centers:
        if not np.datetime64("1959-01-01") < np.datetime64(center) < np.datetime64("2023-01-01"):
            raise ValueError("Choose dates inside the public weather dataset (1959–2022).")

    identity = dict(graphcast_commit=GRAPHCAST_COMMIT, checkpoint=MODEL_NAME,
                    weather_store=WEATHER_STORE, layer=8, node_set="mesh_nodes",
                    mode="post_res", dtype="float32", shape=[40962, 512])
    manifest_path = directory / "dataset.json"
    if manifest_path.exists():
        old = json.loads(manifest_path.read_text())
        if old.get("source") != identity:
            raise ValueError("This folder contains a different dataset. Choose a new PROJECT_FOLDER.")
    elif list(directory.glob("layer0008_*.npy")):
        raise ValueError("This folder already contains activations of unknown origin. Choose a new PROJECT_FOLDER.")
    manifest = dict(source=identity, requested_centers=centers, completed_centers=[])
    pending = []
    for center in centers:
        path = directory / f"layer0008_t{center}.npy"
        if path.exists():
            check_array(path)
            manifest["completed_centers"].append(center)
            print(f"Already saved; skipping {center}", flush=True)
        else:
            pending.append(center)
    write_json(manifest_path, manifest)
    if not pending:
        print("All requested activations are already in Drive. Open notebook 2.", flush=True)
        return

    if not any(device.platform == "gpu" for device in jax.devices()):
        raise RuntimeError("GraphCast cannot see a GPU. Select an A100 or H100 runtime and rerun this notebook.")
    print("GPU ready:", jax.devices(), flush=True)

    # Public files: no Google Cloud account or API key is needed.
    bucket = storage.Client.create_anonymous_client().bucket("dm_graphcast")
    cache = Path("/content/ams691_graphcast_downloads")
    cache.mkdir(parents=True, exist_ok=True)

    def download(blob_name):
        path = cache / Path(blob_name).name
        if not path.exists():
            print("Downloading", path.name, flush=True)
            temporary = path.with_suffix(path.suffix + ".tmp")
            bucket.blob("graphcast/" + blob_name).download_to_filename(str(temporary))
            temporary.replace(path)
        return path

    with download("params/" + MODEL_NAME).open("rb") as handle:
        ckpt = checkpoint.load(handle, graphcast.CheckPoint)
    if ckpt.model_config.latent_size != 512 or ckpt.model_config.mesh_size != 6:
        raise ValueError("Downloaded model does not match the intended GraphCast configuration.")
    stats = {
        name: xr.load_dataset(download("stats/" + name + ".nc"))
        for name in ["diffs_stddev_by_level", "mean_by_level", "stddev_by_level"]
    }

    print("Opening the public weather dataset...", flush=True)
    source = xr.open_zarr(WEATHER_STORE, consolidated=True, storage_options={"token": "anon"})
    source = source.rename({name: new for name, new in [("latitude", "lat"), ("longitude", "lon")]
                            if name in source.coords})
    if source.lat.values[0] > source.lat.values[-1]:
        source = source.isel(lat=slice(None, None, -1))
    needed = set(ckpt.task_config.input_variables + ckpt.task_config.target_variables + ckpt.task_config.forcing_variables)
    needed -= set(graphcast.GENERATED_FORCING_VARS)
    missing = needed - set(source.data_vars)
    if missing:
        raise ValueError(f"Weather dataset is missing {sorted(missing)}")
    source = source[sorted(needed)]
    if source.sizes["lat"] != 721 or source.sizes["lon"] != 1440:
        raise ValueError("Weather grid is not the expected global 0.25-degree grid.")

    manager = get_activation_manager()
    install_writer(manager, directory)
    forward = make_forward(ckpt.model_config, ckpt.task_config, stats)
    # Passing weights as arguments avoids embedding a large constant in the executable.
    predict = jax.jit(forward.apply)

    for number, center in enumerate(pending, 1):
        print(f"\nSnapshot {number}/{len(pending)}: {center} UTC", flush=True)
        print("Reading three weather times (this can take several minutes)...", flush=True)
        window = forecast_window(source, center, ckpt.task_config.pressure_levels)
        inputs, targets, forcings = data_utils.extract_inputs_targets_forcings(
            window, target_lead_times=slice("6h", "6h"), **dataclasses.asdict(ckpt.task_config)
        )
        manager.set_time(center)
        print("Running GraphCast; the first forecast also compiles the model...", flush=True)
        result = predict(ckpt.params, {}, jax.random.PRNGKey(0), inputs, targets * np.nan, forcings)
        jax.block_until_ready(result)
        # Finish every saving callback before changing its timestamp.
        jax.effects_barrier()
        check_array(directory / f"layer0008_t{center}.npy")
        manifest["completed_centers"].append(center)
        write_json(manifest_path, manifest)
        del result, window, inputs, targets, forcings
        gc.collect()

    source.close()
    print(f"\nFinished: {len(centers)} snapshots are saved in {directory}", flush=True)
    print("You can disconnect this runtime and open notebook 2.", flush=True)


if __name__ == "__main__":
    main(sys.argv[1])

## Step 5 — Run GraphCast and save your activations

This is the cell that **actually runs GraphCast**. Each completed snapshot is saved immediately to Drive.
If Colab disconnects, reopen this notebook and run it again; completed files will be skipped.

In [ ]:
# @title Extract and save to Drive
process = subprocess.Popen(
    [PYTHON, '-u', '/content/ams691_extract.py', str(CONFIG_PATH)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    env={**os.environ, 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false'},
)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
except BaseException:
    process.terminate()
    process.wait()
    raise
if returncode:
    raise RuntimeError('Extraction stopped. Completed snapshots are safe in Drive. Read the error above.')

manifest = json.loads((ACTIVATION_DIR / 'dataset.json').read_text())
assert set(manifest['requested_centers']) <= set(manifest['completed_centers'])
print('\nREADY FOR NOTEBOOK 2: your activations are saved in Google Drive.')

## Next

Choose **Runtime → Disconnect and delete runtime** to release the large GPU. Your Drive files will stay saved.
Then open [**2. Train TopK SAE**](https://colab.research.google.com/github/observerhan/graphcast-interpretability/blob/simple-colab/02_train_topk_sae.ipynb). A T4 GPU is sufficient for that notebook.

Based on the authors' [hurricane demo](https://github.com/theodoremacmillan/graphcast-interpretability/blob/main/hurricane_feature_demo.ipynb) and [GraphCast hooks](https://github.com/theodoremacmillan/graphcast/tree/sae-hooks). Public weather data are from [WeatherBench 2](https://weatherbench2.readthedocs.io/en/latest/data-guide.html).